# Zapdos Cascade Demo — Motion-Gated YOLOv8n (Self-Contained)

**Goal:** measure how much cheaper continuous CCTV inference gets when
you put a `cv2.absdiff` motion gate in front of YOLOv8n.

**How to run:**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. Add your Roboflow API key to Colab Secrets as `ROBOFLOW_API_KEY`
   (left sidebar -> 🔑 icon -> Add new secret, toggle notebook access ON).
3. `Runtime -> Run all`. End-to-end runtime is ~10-15 minutes on a T4.

This notebook is fully self-contained — no cloning, no source-location
dependency. All the `motion_gate` / `detector` / `cascade` code lives in
cells below.

## 1. Install dependencies

In [ ]:
# Pinned versions so this doesn't break when Ultralytics ships v9.
# -q keeps the log short.
!pip install -q 'ultralytics>=8.2.0' 'roboflow>=1.1.0' 'opencv-python-headless>=4.9.0' 'numpy>=1.24.0' 'pandas>=2.0.0' 'pyyaml>=6.0'

## 2. Inline the motion gate

Compares consecutive frames by mean absolute pixel difference in
grayscale. If the difference is below a threshold, the caller skips
expensive inference.

Cost of this check on a 640x480 frame: ~100 microseconds on CPU.
Cost of YOLOv8n inference on the same frame: several milliseconds on
GPU. Every skipped frame is a saved GPU cycle at ~50x cost ratio.

In [ ]:
import cv2

def motion_score(current, previous):
    """Mean absolute grayscale pixel difference between two frames."""
    # First frame: nothing to compare against, so force it through the
    # gate. 999.0 is well above any realistic threshold.
    if previous is None:
        return 999.0

    # Grayscale drops 3 channels to 1 — enough for 'did anything move?'
    # and about 3x cheaper than a color diff.
    cur_gray = cv2.cvtColor(current, cv2.COLOR_BGR2GRAY)
    prev_gray = cv2.cvtColor(previous, cv2.COLOR_BGR2GRAY)

    # absdiff = |current - previous| per pixel. Mean over the whole
    # frame gives a single number in [0, 255]: 0 = identical, higher =
    # more change. An empty scene is typically < 1.0; a person walking
    # through pushes it to 5-20+.
    diff = cv2.absdiff(cur_gray, prev_gray)
    return float(diff.mean())

print('motion_gate loaded')

## 3. Inline the detector wrapper

Stock COCO-pretrained YOLOv8n. Knows 80 general classes (person, car
...) but NOT safety-specific classes like NO-Hardhat. That's fine for
the demo — the cascade's cost behavior is model-agnostic, so we don't
need a fine-tuned detector to measure the frame-skipping ratio.

In [ ]:
from ultralytics import YOLO

class Detector:
    """Thin wrapper around Ultralytics YOLO so we have a stable interface."""

    def __init__(self, weights_path='yolov8n.pt', confidence=0.25):
        # Ultralytics auto-downloads yolov8n.pt (~6 MB) on first use.
        self.model = YOLO(weights_path)
        self.confidence = confidence

    def detect(self, frame):
        # verbose=False silences per-frame stdout spam that would
        # otherwise dominate the notebook output.
        results = self.model(frame, verbose=False, conf=self.confidence)
        # YOLO returns a list (one entry per input image); we send one.
        return results[0]

    @property
    def names(self):
        return self.model.names

print('Detector wrapper loaded')

## 4. Inline the harness (baseline + cascade + cost formula)

Three small functions:
- `run_baseline`: detector on every frame (the naive deployment).
- `run_cascade`: motion gate first, detector only on passing frames.
- `cost_per_camera_month`: convert ms/frame to $/camera/month.

In [ ]:
import time

def run_baseline(frames, detector):
    """Detector on every frame — the naive deployment."""
    t0 = time.time()
    detections = 0
    n = 0
    for tag, frame in frames:
        r = detector.detect(frame)
        # r.boxes is a Boxes object; len() = detections above threshold.
        detections += len(r.boxes)
        n += 1
    wall = time.time() - t0
    return {
        'wall_time_s': wall,
        'frames_processed': n,
        'detections_total': detections,
        # Guard against empty input.
        'ms_per_frame': (wall / n) * 1000 if n else 0.0,
    }


def run_cascade(frames, detector, motion_threshold):
    """Motion gate first; detector only on frames that pass."""
    t0 = time.time()
    detections = 0
    n_gated = 0     # total frames the gate saw
    n_detected = 0  # frames that reached the detector
    prev = None

    for tag, frame in frames:
        n_gated += 1
        # motion_score returns 999.0 on the very first frame so it
        # always passes — we can't skip a frame we haven't seen yet.
        if motion_score(frame, prev) > motion_threshold:
            r = detector.detect(frame)
            detections += len(r.boxes)
            n_detected += 1
        # else: in a real deployment, reuse the previous detector
        # output (scene didn't change). For a cost demo we just skip.
        prev = frame

    wall = time.time() - t0
    return {
        'wall_time_s': wall,
        'frames_processed_by_gate': n_gated,
        'frames_through_detector': n_detected,
        'detections_total': detections,
        'ms_per_frame_avg': (wall / n_gated) * 1000 if n_gated else 0.0,
        # Fraction of frames the gate let through. Lower = more savings.
        'gate_pass_rate': n_detected / n_gated if n_gated else 0.0,
    }


def cost_per_camera_month(ms_per_frame, fps=5, gpu_hourly_usd=0.526,
                          hours_per_month=24 * 30):
    """Convert per-frame latency to $/camera/month.

    Formula:
        GPU-seconds per wall-second = (ms_per_frame / 1000) * fps
        GPU-seconds per month        = above * 3600 * hours_per_month
        $ per month                  = (GPU-seconds / 3600) * hourly

    Default: AWS g4dn.xlarge (1x T4), $0.526/hr on-demand.
    """
    # Seconds of GPU we burn per wall-clock second of camera feed.
    gpu_sec_per_wall_sec = (ms_per_frame / 1000) * fps
    gpu_sec_per_month = gpu_sec_per_wall_sec * 3600 * hours_per_month
    # GPU-seconds -> GPU-hours -> dollars.
    return (gpu_sec_per_month / 3600) * gpu_hourly_usd

print('Harness loaded')

## 5. Imports and seed

In [ ]:
import os
import random
import glob
import csv
import numpy as np
import pandas as pd
from pathlib import Path

# Deterministic run so the '15% active' splice pattern is reproducible.
random.seed(42)
np.random.seed(42)
print('Imports OK')

## 6. Download the Roboflow dataset

Roboflow Universe -> **Construction Site Safety v28** (CC BY 4.0, 2,801
images). Download is ~200 MB and takes 1-2 minutes on Colab's network.

Colab Secrets keeps the API key out of the notebook JSON — so this
notebook is safe to share/commit.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
assert ROBOFLOW_API_KEY, 'Set ROBOFLOW_API_KEY in Colab Secrets first (left sidebar 🔑 icon).'

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('roboflow-universe-projects').project('construction-site-safety')
# yolov8 format = the layout YOLO expects if we later fine-tune.
dataset = project.version(28).download('yolov8')

DATA_ROOT = Path(dataset.location)
print('Dataset at:', DATA_ROOT)
print('Splits:', [p.name for p in DATA_ROOT.iterdir() if p.is_dir()])

## 7. Build the synthetic streaming clip

Real labeled continuous CCTV is either behind research agreements
(VIRAT) or unlabeled (YouTube). For a *cost* measurement that's fine —
the GPU spends the same time on any frame regardless of provenance.

We build 1,500 frames = 5 minutes @ 5 fps:
- **85% static frames** = one background image + tiny per-frame noise
  (an empty warehouse aisle has sensor noise even when nothing moves).
- **15% active frames** = a random dataset image spliced in (motion).

Each frame is tagged 'static' or 'active' so we can inspect gate
decisions later.

In [ ]:
TOTAL_FRAMES = 1500       # 5 min at 5 fps
ACTIVE_RATIO = 0.15       # 15% of frames have real motion
FRAME_SHAPE = (480, 640)  # HxW — standard CCTV resolution

train_images = sorted(glob.glob(str(DATA_ROOT / 'train' / 'images' / '*.jpg')))
assert train_images, 'No training images found — check dataset path.'
print(f'Have {len(train_images)} images to draw from')

def load_and_resize(path):
    img = cv2.imread(path)
    return cv2.resize(img, (FRAME_SHAPE[1], FRAME_SHAPE[0]))

# One 'background' scene the static frames vary around.
background = load_and_resize(train_images[0])

frames = []  # list of (tag, frame) tuples — what the harness eats
for i in range(TOTAL_FRAMES):
    if random.random() < ACTIVE_RATIO:
        active = load_and_resize(random.choice(train_images))
        frames.append(('active', active))
    else:
        # Static frame + tiny noise. Perfect duplicates would give the
        # gate a suspiciously easy win. Real CCTV has sensor noise,
        # flicker, and compression artifacts even on 'empty' scenes.
        noise = np.random.randint(-2, 3, background.shape, dtype=np.int16)
        static = np.clip(background.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        frames.append(('static', static))

n_active = sum(1 for tag, _ in frames if tag == 'active')
print(f'Built {len(frames)} frames: {n_active} active ({n_active/len(frames):.0%})')

## 8. Load YOLOv8n and warm up the GPU

First inference call on a fresh GPU includes CUDA init + kernel
compilation (~2-5s). If we don't warm up, those seconds get charged to
whichever run happens first and skew the comparison.

In [ ]:
detector = Detector(weights_path='yolov8n.pt', confidence=0.25)

# Warm-up: 3 real inference calls so CUDA kernels are compiled and
# cudnn has selected its algorithm before we start the stopwatch.
for _ in range(3):
    detector.detect(frames[0][1])
print('Detector ready. Classes:', len(detector.names))

## 9. Baseline: YOLOv8n on every frame

In [ ]:
baseline = run_baseline(frames, detector)
print('Baseline result:')
for k, v in baseline.items():
    print(f'  {k}: {v}')

## 10. Pick a motion threshold

Too low: gate lets everything through, no savings. Too high: misses
real activity, recall drops.

We score all frames once, then sweep thresholds. Pick the highest that
still lets ~all `active` frames through.

In [ ]:
# Score every consecutive pair once; reuse across all thresholds.
scores = []
prev = None
for tag, frame in frames:
    scores.append((tag, motion_score(frame, prev)))
    prev = frame

total_active = sum(1 for tag, _ in scores if tag == 'active')
print(f"{'threshold':>10} {'pass_rate':>10} {'active_kept':>15} {'static_passed':>15}")
for thr in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
    n_pass = sum(1 for _, s in scores if s > thr)
    active_kept = sum(1 for tag, s in scores if tag == 'active' and s > thr)
    static_passed = sum(1 for tag, s in scores if tag == 'static' and s > thr)
    kept_str = f'{active_kept}/{total_active}'
    print(f'{thr:>10.1f} {n_pass/len(scores):>10.2%} {kept_str:>15} {static_passed:>15}')

# Pick the highest threshold that still keeps every 'active' frame.
# Bumping above that starts throwing away real motion. Adjust manually
# if the sweep table above suggests a different sweet spot.
MOTION_THRESHOLD = 2.0
print(f'\nChosen MOTION_THRESHOLD = {MOTION_THRESHOLD}')

## 11. Cascade: motion gate -> YOLOv8n

In [ ]:
cascade = run_cascade(frames, detector, motion_threshold=MOTION_THRESHOLD)
print('Cascade result:')
for k, v in cascade.items():
    print(f'  {k}: {v}')

## 12. Convert timings to $/camera/month

AWS g4dn.xlarge, 1x T4, $0.526/hr on-demand, continuous 24/7 at 5 fps.

In [ ]:
FPS = 5
GPU_HOURLY = 0.526

cost_baseline = cost_per_camera_month(baseline['ms_per_frame'], fps=FPS, gpu_hourly_usd=GPU_HOURLY)
cost_cascade = cost_per_camera_month(cascade['ms_per_frame_avg'], fps=FPS, gpu_hourly_usd=GPU_HOURLY)

speedup = baseline['ms_per_frame'] / cascade['ms_per_frame_avg']
cost_reduction = cost_baseline / cost_cascade

print(f"Baseline: {baseline['ms_per_frame']:.2f} ms/frame  |  ${cost_baseline:.2f}/camera/month")
print(f"Cascade:  {cascade['ms_per_frame_avg']:.2f} ms/frame  |  ${cost_cascade:.2f}/camera/month")
print(f'Speedup:  {speedup:.2f}x')
print(f'Cost reduction: {cost_reduction:.2f}x')

## 13. Spot-check detector recall on labeled test images

Synthetic clip is fine for cost. To sanity-check that the gate isn't
hiding recall problems on real data, we run both configs over the
labeled test split and count how many labeled images still trigger a
detection.

Caveat: stock YOLOv8n only knows COCO classes (person, car, ...). It
will NOT fire on 'NO-Hardhat' as a distinct class — for that you'd
fine-tune. So we only score classes YOLO already knows.

In [ ]:
test_images = sorted(glob.glob(str(DATA_ROOT / 'test' / 'images' / '*.jpg')))[:100]
print(f'Recall check on {len(test_images)} labeled test images')

hits_baseline = 0
hits_cascade = 0
prev = None

for path in test_images:
    img = cv2.imread(path)
    img = cv2.resize(img, (FRAME_SHAPE[1], FRAME_SHAPE[0]))

    # Baseline: always run detector.
    r_base = detector.detect(img)
    if len(r_base.boxes) > 0:
        hits_baseline += 1

    # Cascade: detector only if gate passes.
    if motion_score(img, prev) > MOTION_THRESHOLD:
        r_cas = detector.detect(img)
        if len(r_cas.boxes) > 0:
            hits_cascade += 1
    prev = img

recall_baseline = hits_baseline / len(test_images)
recall_cascade = hits_cascade / len(test_images)
print(f'Baseline recall: {recall_baseline:.2%} ({hits_baseline}/{len(test_images)})')
print(f'Cascade recall:  {recall_cascade:.2%} ({hits_cascade}/{len(test_images)})')
print(f'Recall delta:    {(recall_cascade - recall_baseline):+.2%}')

## 14. Save results to `summary.csv`

Every assumption goes in the CSV so it's obvious what the numbers are
conditional on.

In [ ]:
os.makedirs('results', exist_ok=True)
rows = [
    ('metric', 'value'),
    ('baseline_ms_per_frame', round(baseline['ms_per_frame'], 2)),
    ('cascade_ms_per_frame', round(cascade['ms_per_frame_avg'], 2)),
    ('baseline_total_seconds', round(baseline['wall_time_s'], 2)),
    ('cascade_total_seconds', round(cascade['wall_time_s'], 2)),
    ('speedup', round(speedup, 2)),
    ('motion_threshold', MOTION_THRESHOLD),
    ('frames_through_detector_baseline', baseline['frames_processed']),
    ('frames_through_detector_cascade', cascade['frames_through_detector']),
    ('cost_baseline_per_camera_month', round(cost_baseline, 2)),
    ('cost_cascade_per_camera_month', round(cost_cascade, 2)),
    ('cost_reduction_x', round(cost_reduction, 2)),
    ('total_frames_in_clip', TOTAL_FRAMES),
    ('fps_assumed', FPS),
    ('gpu_hourly_usd', GPU_HOURLY),
    ('static_ratio_assumed', 1 - ACTIVE_RATIO),
]

with open('results/summary.csv', 'w', newline='') as f:
    csv.writer(f).writerows(rows)

print('Saved results/summary.csv:')
print(open('results/summary.csv').read())

# Uncomment to download the CSV to your local machine:
# from google.colab import files
# files.download('results/summary.csv')

## 15. Headline — copy these numbers into README + writeup

In [ ]:
print('=' * 60)
print('HEADLINE')
print('=' * 60)
print(f"Baseline:       {baseline['ms_per_frame']:6.2f} ms/frame   ${cost_baseline:7.2f}/camera/month")
print(f"Cascade:        {cascade['ms_per_frame_avg']:6.2f} ms/frame   ${cost_cascade:7.2f}/camera/month")
print(f'Cost reduction: {cost_reduction:.2f}x')
print(f'Motion threshold used: {MOTION_THRESHOLD}')
print(f"Gate pass rate:        {cascade['gate_pass_rate']:.2%}")
print('=' * 60)